In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.metrics import confusion_matrix, classification_report

pd.set_option("display.max_columns", 120)
DATA_DIR = "../data/"

model_df = pd.read_parquet(DATA_DIR + "train_features.parquet")
print("Loaded:", model_df.shape, "| LightGBM", lgb.__version__)

Loaded: (1122452, 214) | LightGBM 4.7.0


In [5]:
model_df.head(3)

,vehicle_id,time_step,class_label,171_0,666_0,427_0,837_0,309_0,835_0,370_0,100_0,171_0_rate,666_0_rate,427_0_rate,837_0_rate,309_0_rate,835_0_rate,370_0_rate,100_0_rate,171_0_reset,666_0_reset,427_0_reset,837_0_reset,309_0_reset,835_0_reset,370_0_reset,100_0_reset,167_0_share,167_1_share,167_2_share,167_3_share,167_4_share,167_5_share,167_6_share,167_7_share,167_8_share,167_9_share,272_0_share,272_1_share,272_2_share,272_3_share,272_4_share,272_5_share,272_6_share,272_7_share,272_8_share,272_9_share,291_0_share,291_1_share,291_2_share,291_3_share,291_4_share,291_5_share,291_6_share,291_7_share,291_8_share,291_9_share,291_10_share,158_0_share,158_1_share,...,Spec_1=Cat8,Spec_1=Cat9,Spec_2=Cat0,Spec_2=Cat1,Spec_2=Cat10,Spec_2=Cat11,Spec_2=Cat12,Spec_2=Cat13,Spec_2=Cat14,Spec_2=Cat15,Spec_2=Cat16,Spec_2=Cat17,Spec_2=Cat18,Spec_2=Cat19,Spec_2=Cat2,Spec_2=Cat20,Spec_2=Cat3,Spec_2=Cat4,Spec_2=Cat5,Spec_2=Cat6,Spec_2=Cat7,Spec_2=Cat8,Spec_2=Cat9,Spec_3=Cat0,Spec_3=Cat1,Spec_3=Cat2,Spec_3=Cat3,Spec_4=Cat0,Spec_4=Cat1,Spec_5=Cat0,Spec_5=Cat1,Spec_5=Cat2,Spec_5=Cat3,Spec_5=Cat4,Spec_6=Cat0,Spec_6=Cat1,Spec_6=Cat10,Spec_6=Cat11,Spec_6=Cat12,Spec_6=Cat13,Spec_6=Cat15,Spec_6=Cat17,Spec_6=Cat18,Spec_6=Cat2,Spec_6=Cat3,Spec_6=Cat4,Spec_6=Cat5,Spec_6=Cat6,Spec_6=Cat7,Spec_6=Cat8,Spec_6=Cat9,Spec_7=Cat0,Spec_7=Cat1,Spec_7=Cat2,Spec_7=Cat3,Spec_7=Cat4,Spec_7=Cat5,Spec_7=Cat6,Spec_7=Cat7,Spec_7=Cat8
0,0,11.2,0,167985.0,10787.0,7413813.0,2296.0,70.0,8036751.0,0.0,858410.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0,0,0,0,0,0,0,0,0.000328,0.103409,0.129879,0.050280,0.101264,0.380715,0.215901,0.017726,0.000498,0.0,0.130602,0.078053,0.034999,0.060851,0.658875,0.036265,0.000354,0.0,0.0,0.0,0.197871,0.089502,0.074665,0.149169,0.075472,0.036284,0.086276,0.083212,0.079342,0.117562,0.010643,0.009569,0.265214,...,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False
1,0,11.4,0,167985.0,10787.0,7413813.0,2296.0,70.0,8040811.0,0.0,860571.0,0.000000,0.000000,0.000000,0.000000,0.0,20300.000000,0.0,10805.000000,0,0,0,0,0,0,0,0,0.000328,0.103869,0.129812,0.050254,0.101212,0.380520,0.215790,0.017717,0.000497,0.0,0.131043,0.078013,0.034982,0.060820,0.658541,0.036247,0.000354,0.0,0.0,0.0,0.198068,0.089855,0.074557,0.148953,0.075523,0.036393,0.086151,0.083092,0.079388,0.117391,0.010628,0.009564,0.265380,...,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False
2,0,19.6,0,331635.0,14525.0,13683604.0,2600.0,70.0,12777022.0,0.0,1379191.0,19957.317073,455.853659,764608.658537,37.073171,0.0,577586.707317,0.0,63246.341463,0,0,0,0,0,0,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.102363,0.064882,0.034261,0.066824,0.705095,0.026353,0.000223,0.0,0.0,0.0,0.215084,0.096063,0.085591,0.142987,0.072702,0.041486,0.088611,0.067063,0.059007,0.115094,0.016313,0.011267,0.263567,...,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False


In [3]:
counter_cols = ["171_0", "666_0", "427_0", "837_0", "309_0", "835_0", "370_0", "100_0"]
rate_cols    = [f"{c}_rate"  for c in counter_cols]
reset_cols   = [f"{c}_reset" for c in counter_cols]
share_cols   = [c for c in model_df.columns if c.endswith("_share")]
spec_cols    = [c for c in model_df.columns if c.startswith("Spec_")]
feature_cols = counter_cols + rate_cols + reset_cols + share_cols + spec_cols + ["time_step"]

COST = np.array([
    [  0,   7,   8,   9,  10],
    [200,   0,   7,   8,   9],
    [300, 200,   0,   7,   8],
    [400, 300, 200,   0,   7],
    [500, 400, 300, 200,   0],
])

def total_cost(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0,1,2,3,4])
    return int((cm * COST).sum())

def cost_report(y_true, y_pred, name):
    tc = total_cost(y_true, y_pred)
    print(f"{name:<38} Total_cost = {tc:>10,}   (per readout: {tc/len(y_true):.2f})")
    return tc

print("Features:", len(feature_cols))

Features: 212


In [4]:
rng = np.random.default_rng(42)
vehicles = model_df["vehicle_id"].unique()
rng.shuffle(vehicles)

n_val = int(0.2 * len(vehicles))
val_vehicles = set(vehicles[:n_val])
is_val = model_df["vehicle_id"].isin(val_vehicles)

train_df, val_df = model_df[~is_val], model_df[is_val]

X_train = train_df[feature_cols].replace([np.inf, -np.inf], np.nan)
y_train = train_df["class_label"].values
X_val   = val_df[feature_cols].replace([np.inf, -np.inf], np.nan)
y_val   = val_df["class_label"].values

print(f"train {X_train.shape} | val {X_val.shape}")

train (898531, 212) | val (223921, 212)


In [4]:
X_train.head(3)

,171_0,666_0,427_0,837_0,309_0,835_0,370_0,100_0,171_0_rate,666_0_rate,427_0_rate,837_0_rate,309_0_rate,835_0_rate,370_0_rate,100_0_rate,171_0_reset,666_0_reset,427_0_reset,837_0_reset,309_0_reset,835_0_reset,370_0_reset,100_0_reset,167_0_share,167_1_share,167_2_share,167_3_share,167_4_share,167_5_share,167_6_share,167_7_share,167_8_share,167_9_share,272_0_share,272_1_share,272_2_share,272_3_share,272_4_share,272_5_share,272_6_share,272_7_share,272_8_share,272_9_share,291_0_share,291_1_share,291_2_share,291_3_share,291_4_share,291_5_share,291_6_share,291_7_share,291_8_share,291_9_share,291_10_share,158_0_share,158_1_share,158_2_share,158_3_share,158_4_share,...,Spec_1=Cat9,Spec_2=Cat0,Spec_2=Cat1,Spec_2=Cat10,Spec_2=Cat11,Spec_2=Cat12,Spec_2=Cat13,Spec_2=Cat14,Spec_2=Cat15,Spec_2=Cat16,Spec_2=Cat17,Spec_2=Cat18,Spec_2=Cat19,Spec_2=Cat2,Spec_2=Cat20,Spec_2=Cat3,Spec_2=Cat4,Spec_2=Cat5,Spec_2=Cat6,Spec_2=Cat7,Spec_2=Cat8,Spec_2=Cat9,Spec_3=Cat0,Spec_3=Cat1,Spec_3=Cat2,Spec_3=Cat3,Spec_4=Cat0,Spec_4=Cat1,Spec_5=Cat0,Spec_5=Cat1,Spec_5=Cat2,Spec_5=Cat3,Spec_5=Cat4,Spec_6=Cat0,Spec_6=Cat1,Spec_6=Cat10,Spec_6=Cat11,Spec_6=Cat12,Spec_6=Cat13,Spec_6=Cat15,Spec_6=Cat17,Spec_6=Cat18,Spec_6=Cat2,Spec_6=Cat3,Spec_6=Cat4,Spec_6=Cat5,Spec_6=Cat6,Spec_6=Cat7,Spec_6=Cat8,Spec_6=Cat9,Spec_7=Cat0,Spec_7=Cat1,Spec_7=Cat2,Spec_7=Cat3,Spec_7=Cat4,Spec_7=Cat5,Spec_7=Cat6,Spec_7=Cat7,Spec_7=Cat8,time_step
0,167985.0,10787.0,7413813.0,2296.0,70.0,8036751.0,0.0,858410.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0,0,0,0,0,0,0,0,0.000328,0.103409,0.129879,0.050280,0.101264,0.380715,0.215901,0.017726,0.000498,0.0,0.130602,0.078053,0.034999,0.060851,0.658875,0.036265,0.000354,0.0,0.0,0.0,0.197871,0.089502,0.074665,0.149169,0.075472,0.036284,0.086276,0.083212,0.079342,0.117562,0.010643,0.009569,0.265214,0.290376,0.077733,0.067779,...,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,11.2
1,167985.0,10787.0,7413813.0,2296.0,70.0,8040811.0,0.0,860571.0,0.000000,0.000000,0.000000,0.000000,0.0,20300.000000,0.0,10805.000000,0,0,0,0,0,0,0,0,0.000328,0.103869,0.129812,0.050254,0.101212,0.380520,0.215790,0.017717,0.000497,0.0,0.131043,0.078013,0.034982,0.060820,0.658541,0.036247,0.000354,0.0,0.0,0.0,0.198068,0.089855,0.074557,0.148953,0.075523,0.036393,0.086151,0.083092,0.079388,0.117391,0.010628,0.009564,0.265380,0.290436,0.077693,0.067745,...,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,11.4
2,331635.0,14525.0,13683604.0,2600.0,70.0,12777022.0,0.0,1379191.0,19957.317073,455.853659,764608.658537,37.073171,0.0,577586.707317,0.0,63246.341463,0,0,0,0,0,0,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.102363,0.064882,0.034261,0.066824,0.705095,0.026353,0.000223,0.0,0.0,0.0,0.215084,0.096063,0.085591,0.142987,0.072702,0.041486,0.088611,0.067063,0.059007,0.115094,0.016313,0.011267,0.263567,0.287227,0.095473,0.088155,...,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,19.6


In [6]:
cost_report(y_val, np.zeros_like(y_val),   "Always predict 0")
cost_report(y_val, np.full_like(y_val, 4), "Always predict 4")

Always predict 0                       Total_cost =  1,489,900   (per readout: 6.65)
Always predict 4                       Total_cost =  2,224,862   (per readout: 9.94)


2224862

In [5]:
out = pd.DataFrame({
    "vehicle_id": val_df.sort_values("time_step").groupby("vehicle_id").tail(1)["vehicle_id"].values,
    "xgb_p4": pv[:, 4],
    "xgb_p_any": pv[:, 1:].sum(axis=1),
    "class_label": yl,
})
out.to_csv("../models/xgb_val_scores.csv", index=False)
print("Saved:", out.shape)

NameError: name 'pv' is not defined

In [6]:
# Check what's available
for v in ["final", "Xl", "yl", "ylt", "train_prior", "val_df", "feature_cols"]:
    print(f"{v:<14}", "OK" if v in dir() else "MISSING")

final          MISSING
Xl             MISSING
yl             MISSING
ylt            MISSING
train_prior    MISSING
val_df         OK
feature_cols   OK


In [7]:
import joblib, numpy as np, pandas as pd
from sklearn.metrics import confusion_matrix

final = joblib.load("../models/xgb_final.pkl")

# Rebuild validation last-readouts
lastv = val_df.sort_values("time_step").groupby("vehicle_id").tail(1)
Xl = lastv[feature_cols].replace([np.inf, -np.inf], np.nan)
yl = lastv["class_label"].values
print("Xl:", Xl.shape, "| class-4:", (yl == 4).sum())

Xl: (4710, 212) | class-4: 404


In [8]:
model_df = pd.read_parquet("../data/train_features.parquet")

rng = np.random.default_rng(42)
vehicles = model_df["vehicle_id"].unique()
rng.shuffle(vehicles)
val_vehicles = set(vehicles[:int(0.2 * len(vehicles))])

train_df = model_df[~model_df["vehicle_id"].isin(val_vehicles)]
last_train = train_df.sort_values("time_step").groupby("vehicle_id").tail(1)
ylt = last_train["class_label"].values

train_prior = pd.Series(ylt).value_counts(normalize=True).sort_index().values
val_prior   = pd.Series(yl).value_counts(normalize=True).sort_index().values
print("train_prior:", train_prior.round(4))
print("val_prior:  ", val_prior.round(4))

train_prior: [0.9038 0.0012 0.0031 0.0072 0.0847]
val_prior:   [0.9047 0.0017 0.0023 0.0055 0.0858]


In [ ]:
pv = final.predict_proba(Xl)
pv = pv * (val_prior / train_prior)
pv = pv / pv.sum(axis=1, keepdims=True)

pd.DataFrame({
    "vehicle_id": lastv["vehicle_id"].values,
    "xgb_p4": pv[:, 4],
    "xgb_p_any": pv[:, 1:].sum(axis=1),
    "class_label": yl,
}).to_csv("../models/xgb_val_scores.csv", index=False)
print("Saved.")

Saved.


: 

In [7]:
params = dict(
    objective="multiclass",
    num_class=5,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=100,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=1,
    n_estimators=1500,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

clf = lgb.LGBMClassifier(**params)
clf.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="multi_logloss",
    callbacks=[lgb.early_stopping(100, verbose=True), lgb.log_evaluation(100)],
)
print("Best iteration:", clf.best_iteration_)

d:\AIPM_Bootcamp\CAE_Projects\engineering-ai-digital-twin-predictive-maintenance\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.123398
Early stopping, best iteration is:
[65]	valid_0's multi_logloss: 0.123302
Best iteration: 65


In [8]:
proba = clf.predict_proba(X_val)

pred_argmax  = proba.argmax(axis=1)
pred_mincost = (proba @ COST).argmin(axis=1)

cost_report(y_val, pred_argmax,  "LightGBM (argmax)")
cost_report(y_val, pred_mincost, "LightGBM (min expected cost)")

print("\nPrediction distribution (min cost):")
print(pd.Series(pred_mincost).value_counts().sort_index())

print("\nConfusion matrix (min cost):")
print(pd.DataFrame(confusion_matrix(y_val, pred_mincost, labels=[0,1,2,3,4]),
                   index=[f"actual {i}" for i in range(5)],
                   columns=[f"pred {i}" for i in range(5)]))

print("\n", classification_report(y_val, pred_mincost, zero_division=0))

LightGBM (argmax)                      Total_cost =  1,482,938   (per readout: 6.62)
LightGBM (min expected cost)           Total_cost =  1,003,041   (per readout: 4.48)

Prediction distribution (min cost):
0    184276
1       196
2      1849
3      6781
4     30819
Name: count, dtype: int64

Confusion matrix (min cost):
          pred 0  pred 1  pred 2  pred 3  pred 4
actual 0  182115     187    1745    6413   28356
actual 1    1156       9      68     196    1101
actual 2     502       0      15      73     630
actual 3     228       0      12      43     313
actual 4     275       0       9      56     419

               precision    recall  f1-score   support

           0       0.99      0.83      0.90    218816
           1       0.05      0.00      0.01      2530
           2       0.01      0.01      0.01      1220
           3       0.01      0.07      0.01       596
           4       0.01      0.55      0.03       759

    accuracy                           0.82    223921
 

In [9]:
print("Predicted mean probability per class:")
print(pd.Series(proba.mean(axis=0), index=range(5)).round(4))
print("\nActual validation frequency:")
print(pd.Series(y_val).value_counts(normalize=True).sort_index().round(4))

Predicted mean probability per class:
0    0.9782
1    0.0101
2    0.0052
3    0.0029
4    0.0036
dtype: float64

Actual validation frequency:
0    0.9772
1    0.0113
2    0.0054
3    0.0027
4    0.0034
Name: proportion, dtype: float64


In [10]:
cost_report(y_val, np.zeros_like(y_val),   "Always predict 0")
cost_report(y_val, np.full_like(y_val, 4), "Always predict 4")

Always predict 0                       Total_cost =  1,489,900   (per readout: 6.65)
Always predict 4                       Total_cost =  2,224,862   (per readout: 9.94)


2224862

In [11]:
last = (val_df.sort_values("time_step")
              .groupby("vehicle_id").tail(1))
Xl = last[feature_cols].replace([np.inf, -np.inf], np.nan)
yl = last["class_label"].values

pl = clf.predict_proba(Xl)
cost_report(yl, pl.argmax(axis=1),        "Per-vehicle last readout (argmax)")
cost_report(yl, (pl @ COST).argmin(axis=1), "Per-vehicle last readout (min cost)")
print(pd.DataFrame(confusion_matrix(yl, (pl @ COST).argmin(axis=1), labels=[0,1,2,3,4]),
                   index=[f"actual {i}" for i in range(5)], columns=[f"pred {i}" for i in range(5)]))

Per-vehicle last readout (argmax)      Total_cost =    216,218   (per readout: 45.91)
Per-vehicle last readout (min cost)    Total_cost =    103,543   (per readout: 21.98)
          pred 0  pred 1  pred 2  pred 3  pred 4
actual 0    2874       1      21     172    1193
actual 1       4       0       0       1       3
actual 2       1       0       0       2       8
actual 3      14       0       1       0      11
actual 4     151       0       5      29     219


In [12]:
import xgboost as xgb

xgb_clf = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=5,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=20,
    subsample=0.8,
    colsample_bytree=0.8,
    n_estimators=1500,
    early_stopping_rounds=100,
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
)

xgb_clf.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)
print("Best iteration:", xgb_clf.best_iteration)

[0]	validation_0-mlogloss:0.13545
[100]	validation_0-mlogloss:0.12167
[200]	validation_0-mlogloss:0.12131
[297]	validation_0-mlogloss:0.12153
Best iteration: 197


In [14]:
!pip install xgboost catboost

   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.8/100.2 MB 6.7 MB/s eta 0:00:15
    --------------------------------------- 2.1/100.2 MB 6.9 MB/s eta 0:00:15
   - -------------------------------------- 3.7/100.2 MB 7.0 MB/s eta 0:00:14
   -- ------------------------------------- 5.2/100.2 MB 7.2 MB/s eta 0:00:14
   -- ------------------------------------- 7.3/100.2 MB 7.8 MB/s eta 0:00:12
   --- ------------------------------------ 8.9/100.2 MB 7.7 MB/s eta 0:00:12
   ---- ----------------------------------- 11.0/100.2 MB 8.1 MB/s eta 0:00:12
   ----- ---------------------------------- 12.6/100.2 MB 8.0 MB/s eta 0:00:11
   ----- ---------------------------------- 14.7/100.2 MB 8.2 MB/s eta 0:00:11
   ------ --------------------------------- 16.5/100.2 MB 8.4 MB/s eta 0:00:10
   ------- -------------------------------- 18.4/100.2 MB 8.5 MB/s eta 0:00:10
   ------- -------------------------------- 19.7/100.2 MB 8.2 MB/s

In [15]:
from catboost import CatBoostClassifier

cat_clf = CatBoostClassifier(
    loss_function="MultiClass",
    classes_count=5,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3,
    iterations=1500,
    early_stopping_rounds=100,
    random_seed=42,
    verbose=100,
)

cat_clf.fit(X_train, y_train, eval_set=(X_val, y_val))
print("Best iteration:", cat_clf.get_best_iteration())

0:	learn: 1.4258487	test: 1.4258208	best: 1.4258208 (0)	total: 596ms	remaining: 14m 53s
100:	learn: 0.1247694	test: 0.1271048	best: 0.1271048 (100)	total: 38.8s	remaining: 8m 57s
200:	learn: 0.1158978	test: 0.1234818	best: 0.1234818 (200)	total: 1m 10s	remaining: 7m 35s
300:	learn: 0.1114432	test: 0.1230702	best: 0.1230469 (286)	total: 1m 39s	remaining: 6m 37s
400:	learn: 0.1075744	test: 0.1227537	best: 0.1227106 (384)	total: 2m 10s	remaining: 5m 56s
500:	learn: 0.1041476	test: 0.1225020	best: 0.1225020 (500)	total: 2m 40s	remaining: 5m 20s
600:	learn: 0.1012231	test: 0.1224083	best: 0.1223403 (543)	total: 3m 10s	remaining: 4m 45s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.1223403367
bestIteration = 543

Shrink model to first 544 iterations.
Best iteration: 543


In [16]:
def evaluate(name, proba, y_true):
    a = proba.argmax(axis=1)
    m = (proba @ COST).argmin(axis=1)
    return {
        "model": name,
        "argmax": total_cost(y_true, a),
        "min_cost": total_cost(y_true, m),
        "class4_recall": (confusion_matrix(y_true, m, labels=[0,1,2,3,4])[4, 4]
                          / max((y_true == 4).sum(), 1)),
    }

proba_lgb = clf.predict_proba(X_val)
proba_xgb = xgb_clf.predict_proba(X_val)
proba_cat = cat_clf.predict_proba(X_val)

rows = [
    evaluate("LightGBM", proba_lgb, y_val),
    evaluate("XGBoost",  proba_xgb, y_val),
    evaluate("CatBoost", proba_cat, y_val),
]

results = pd.DataFrame(rows)
results["improvement_vs_always0"] = (1 - results["min_cost"] / 1_489_900).round(3)
print(results.to_string(index=False))

   model  argmax  min_cost  class4_recall  improvement_vs_always0
LightGBM 1482938   1003041       0.552042                   0.327
 XGBoost 1488344    966546       0.674572                   0.351
CatBoost 1488841   1012692       0.649539                   0.320


In [17]:
last = val_df.sort_values("time_step").groupby("vehicle_id").tail(1)
Xl = last[feature_cols].replace([np.inf, -np.inf], np.nan)
yl = last["class_label"].values

rows_v = [
    evaluate("LightGBM", clf.predict_proba(Xl), yl),
    evaluate("XGBoost",  xgb_clf.predict_proba(Xl), yl),
    evaluate("CatBoost", cat_clf.predict_proba(Xl), yl),
]
print(pd.DataFrame(rows_v).to_string(index=False))

   model  argmax  min_cost  class4_recall
LightGBM  216218    103543       0.542079
 XGBoost  216800     86742       0.660891
CatBoost  217300     91575       0.643564


In [18]:
actual = pd.Series(y_val).value_counts(normalize=True).sort_index()
calib = pd.DataFrame({
    "actual":   actual.values,
    "LightGBM": proba_lgb.mean(axis=0),
    "XGBoost":  proba_xgb.mean(axis=0),
    "CatBoost": proba_cat.mean(axis=0),
}, index=[f"class {i}" for i in range(5)])
print(calib.round(4))

         actual  LightGBM  XGBoost  CatBoost
class 0  0.9772    0.9782   0.9778    0.9769
class 1  0.0113    0.0101   0.0106    0.0110
class 2  0.0054    0.0052   0.0054    0.0055
class 3  0.0027    0.0029   0.0029    0.0030
class 4  0.0034    0.0036   0.0036    0.0036


In [19]:
from itertools import product

def cost_at_vehicle_level(model):
    p = model.predict_proba(Xl)
    return total_cost(yl, (p @ COST).argmin(axis=1))

results = []
for depth, mcw, lr in product([4, 6, 8], [1, 20, 100], [0.05, 0.1]):
    m = xgb.XGBClassifier(
        objective="multi:softprob", num_class=5,
        max_depth=depth, min_child_weight=mcw, learning_rate=lr,
        subsample=0.8, colsample_bytree=0.8,
        n_estimators=1000, early_stopping_rounds=50,
        eval_metric="mlogloss", tree_method="hist",
        random_state=42, n_jobs=-1,
    )
    m.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    c = cost_at_vehicle_level(m)
    results.append({"depth": depth, "min_child_weight": mcw, "lr": lr,
                    "best_iter": m.best_iteration, "vehicle_cost": c})
    print(f"depth={depth} mcw={mcw} lr={lr} -> {c:,}")

print(pd.DataFrame(results).sort_values("vehicle_cost").to_string(index=False))

depth=4 mcw=1 lr=0.05 -> 87,542
depth=4 mcw=1 lr=0.1 -> 88,634
depth=4 mcw=20 lr=0.05 -> 86,684
depth=4 mcw=20 lr=0.1 -> 88,527
depth=4 mcw=100 lr=0.05 -> 85,061
depth=4 mcw=100 lr=0.1 -> 86,239
depth=6 mcw=1 lr=0.05 -> 88,661
depth=6 mcw=1 lr=0.1 -> 90,442
depth=6 mcw=20 lr=0.05 -> 86,742
depth=6 mcw=20 lr=0.1 -> 87,570
depth=6 mcw=100 lr=0.05 -> 85,820
depth=6 mcw=100 lr=0.1 -> 85,106
depth=8 mcw=1 lr=0.05 -> 95,853
depth=8 mcw=1 lr=0.1 -> 99,507
depth=8 mcw=20 lr=0.05 -> 89,614
depth=8 mcw=20 lr=0.1 -> 90,163
depth=8 mcw=100 lr=0.05 -> 87,537
depth=8 mcw=100 lr=0.1 -> 89,609
 depth  min_child_weight   lr  best_iter  vehicle_cost
     4               100 0.05        352         85061
     6               100 0.10        101         85106
     6               100 0.05        182         85820
     4               100 0.10        168         86239
     4                20 0.05        338         86684
     6                20 0.05        197         86742
     8               100 0.05 

In [20]:
best = xgb.XGBClassifier(
    objective="multi:softprob", num_class=5,
    max_depth=4, min_child_weight=100, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    n_estimators=1000, early_stopping_rounds=50,
    eval_metric="mlogloss", tree_method="hist",
    random_state=42, n_jobs=-1,
)
best.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

import joblib
joblib.dump(best, "../models/xgb_tuned.pkl")

pv = best.predict_proba(Xl)
print("Vehicle-level:", f"{total_cost(yl, (pv @ COST).argmin(axis=1)):,}")
print("Best iteration:", best.best_iteration)

Vehicle-level: 85,061
Best iteration: 352


In [21]:
# Work on the full readout table, sorted by vehicle and time
ops = model_df.sort_values(["vehicle_id", "time_step"]).copy()
g = ops.groupby("vehicle_id")

trend_parts = []

# --- 1. Recent vs lifetime usage intensity ---
# Has this truck's usage rate accelerated relative to its own normal?
for c in rate_cols:
    recent   = g[c].transform(lambda s: s.rolling(3, min_periods=1).mean())
    lifetime = g[c].transform(lambda s: s.expanding().mean())
    ops[f"{c}_ratio"] = recent / lifetime.replace(0, np.nan)

ratio_cols = [f"{c}_ratio" for c in rate_cols]

# --- 2. Histogram profile drift vs the vehicle's own early baseline ---
# Baseline = mean profile over that vehicle's first 5 readouts
drift_cols = []
for var, cols in [(v, [c for c in share_cols if c.startswith(f"{v}_")])
                  for v in ["167", "272", "291", "158", "459", "397"]]:
    base = g[cols].transform(lambda s: s.head(5).mean() if len(s) else np.nan)
    drift = np.abs(ops[cols].values - base.values).sum(axis=1)   # L1 distance
    ops[f"{var}_drift"] = drift
    drift_cols.append(f"{var}_drift")

# --- 3. Monitoring density: readouts per time unit in recent window ---
ops["readout_gap"]     = g["time_step"].diff()
ops["gap_ratio"]       = ops["readout_gap"] / g["readout_gap"].transform("mean").replace(0, np.nan)

# --- 4. Age relative to fleet: how far into a typical life is this truck? ---
ops["readout_index"]   = g.cumcount()

new_cols = ratio_cols + drift_cols + ["readout_gap", "gap_ratio", "readout_index"]
ops[new_cols] = ops[new_cols].replace([np.inf, -np.inf], np.nan)
ops = ops.copy()

print("New features:", len(new_cols))
print(ops[new_cols].describe().T[["mean", "50%", "max"]].round(3))

New features: 17
                    mean     50%      max
171_0_rate_ratio   1.040   1.045    8.604
666_0_rate_ratio   1.032   1.000   43.501
427_0_rate_ratio   1.039   1.046   12.259
837_0_rate_ratio   1.123   0.990   65.000
309_0_rate_ratio   1.324   0.725   79.676
835_0_rate_ratio   1.042   1.046   12.402
370_0_rate_ratio   1.083   1.000   53.333
100_0_rate_ratio   1.060   1.027   11.974
167_drift          0.188   0.137    1.830
272_drift          0.158   0.100    1.825
291_drift          0.143   0.099    1.668
158_drift          0.100   0.080    1.513
459_drift          0.177   0.133    1.938
397_drift          0.209   0.176    1.784
readout_gap        4.869   4.400  387.800
gap_ratio          1.000   1.020   31.017
readout_index     31.210  26.000  302.000


In [22]:
feature_cols_v2 = feature_cols + new_cols

trainv, valv = ops[~ops["vehicle_id"].isin(val_vehicles)], ops[ops["vehicle_id"].isin(val_vehicles)]
Xt2, yt2 = trainv[feature_cols_v2], trainv["class_label"].values
Xv2, yv2 = valv[feature_cols_v2],  valv["class_label"].values

m2 = xgb.XGBClassifier(
    objective="multi:softprob", num_class=5,
    max_depth=4, min_child_weight=100, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    n_estimators=1000, early_stopping_rounds=50,
    eval_metric="mlogloss", tree_method="hist",
    random_state=42, n_jobs=-1,
)
m2.fit(Xt2, yt2, eval_set=[(Xv2, yv2)], verbose=False)

lastv = valv.sort_values("time_step").groupby("vehicle_id").tail(1)
Xl2, yl2 = lastv[feature_cols_v2], lastv["class_label"].values
p2 = m2.predict_proba(Xl2)
pred2 = (p2 @ COST).argmin(axis=1)

print("Vehicle-level cost:", f"{total_cost(yl2, pred2):,}   (was 85,061)")
print("Class-4 recall:", round(confusion_matrix(yl2, pred2, labels=[0,1,2,3,4])[4,4] / (yl2==4).sum(), 3))
print(pd.DataFrame(confusion_matrix(yl2, pred2, labels=[0,1,2,3,4]),
                   index=[f"actual {i}" for i in range(5)], columns=[f"pred {i}" for i in range(5)]))

Vehicle-level cost: 87,583   (was 85,061)
Class-4 recall: 0.656
          pred 0  pred 1  pred 2  pred 3  pred 4
actual 0    2686       0      10     122    1443
actual 1       5       0       0       0       3
actual 2       2       0       0       1       8
actual 3      13       0       0       2      11
actual 4     124       0       0      15     265


In [23]:
imp = pd.Series(m2.feature_importances_, index=feature_cols_v2).sort_values(ascending=False)
print("Top 20 overall:")
print(imp.head(20).round(4))
print("\nRank of new features:")
print(imp.rank(ascending=False)[new_cols].sort_values().round(0))

Top 20 overall:
666_0               0.0590
427_0               0.0233
835_0               0.0188
Spec_2=Cat5         0.0161
Spec_4=Cat0         0.0156
Spec_1=Cat9         0.0148
427_0_rate          0.0147
Spec_7=Cat6         0.0144
Spec_2=Cat6         0.0135
Spec_2=Cat0         0.0133
Spec_2=Cat10        0.0112
459_14_share        0.0110
Spec_1=Cat1         0.0108
171_0               0.0106
167_5_share         0.0100
309_0               0.0099
837_0_rate_ratio    0.0092
Spec_7=Cat7         0.0091
Spec_1=Cat7         0.0085
167_0_share         0.0084
dtype: float32

Rank of new features:
837_0_rate_ratio     17.0
666_0_rate_ratio     21.0
835_0_rate_ratio     54.0
309_0_rate_ratio    131.0
291_drift           136.0
167_drift           139.0
readout_gap         141.0
158_drift           142.0
397_drift           147.0
readout_index       148.0
272_drift           149.0
459_drift           157.0
171_0_rate_ratio    163.0
370_0_rate_ratio    164.0
427_0_rate_ratio    166.0
gap_ratio       

In [24]:
p_ens = (clf.predict_proba(Xl) + best.predict_proba(Xl) + cat_clf.predict_proba(Xl)) / 3
print("Ensemble:", f"{total_cost(yl, (p_ens @ COST).argmin(axis=1)):,}")

Ensemble: 88,254


In [25]:
last_train = trainv.sort_values("time_step").groupby("vehicle_id").tail(1)
Xlt, ylt = last_train[feature_cols], last_train["class_label"].values
print("Training rows:", len(Xlt), dict(pd.Series(ylt).value_counts().sort_index()))

m3 = xgb.XGBClassifier(
    objective="multi:softprob", num_class=5,
    max_depth=4, min_child_weight=20, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    n_estimators=1000, early_stopping_rounds=50,
    eval_metric="mlogloss", tree_method="hist", random_state=42, n_jobs=-1,
)
m3.fit(Xlt, ylt, eval_set=[(Xl, yl)], verbose=False)
p3 = m3.predict_proba(Xl)
print("Last-readout training:", f"{total_cost(yl, (p3 @ COST).argmin(axis=1)):,}  (was 85,061)")

Training rows: 18840 {0: np.int64(17027), 1: np.int64(23), 2: np.int64(59), 3: np.int64(135), 4: np.int64(1596)}
Last-readout training: 39,002  (was 85,061)


In [26]:
# 1. Calibration - is it still honest, or winning by over-predicting?
print("Predicted mean:", p3.mean(axis=0).round(4))
print("Actual freq:   ", pd.Series(yl).value_counts(normalize=True).sort_index().round(4).values)

# 2. Confusion matrix and recall - where did the gain come from?
pred3 = (p3 @ COST).argmin(axis=1)
cm = confusion_matrix(yl, pred3, labels=[0,1,2,3,4])
print(pd.DataFrame(cm, index=[f"actual {i}" for i in range(5)],
                   columns=[f"pred {i}" for i in range(5)]))
print("Class-4 recall:", round(cm[4,4] / (yl==4).sum(), 3), " (was 0.661)")
print("Missed class-4:", cm[4,0], "vehicles")

Predicted mean: [0.9073 0.0012 0.0028 0.0064 0.0822]
Actual freq:    [0.9047 0.0017 0.0023 0.0055 0.0858]
          pred 0  pred 1  pred 2  pred 3  pred 4
actual 0    1195       0       0       0    3066
actual 1       0       0       0       0       8
actual 2       0       0       0       0      11
actual 3       0       0       0       0      26
actual 4      16       0       0       0     388
Class-4 recall: 0.96  (was 0.661)
Missed class-4: 16 vehicles


In [27]:
from itertools import product

def score(m):
    p = m.predict_proba(Xl)
    pred = (p @ COST).argmin(axis=1)
    cm = confusion_matrix(yl, pred, labels=[0,1,2,3,4])
    return total_cost(yl, pred), cm[4,4] / (yl == 4).sum(), cm[4,0]

rows = []
for depth, mcw, lr in product([3, 4, 5, 6], [1, 5, 10, 20, 50], [0.05, 0.1]):
    m = xgb.XGBClassifier(
        objective="multi:softprob", num_class=5,
        max_depth=depth, min_child_weight=mcw, learning_rate=lr,
        subsample=0.8, colsample_bytree=0.8,
        n_estimators=2000, early_stopping_rounds=100,
        eval_metric="mlogloss", tree_method="hist",
        random_state=42, n_jobs=-1,
    )
    m.fit(Xlt, ylt, eval_set=[(Xl, yl)], verbose=False)
    c, rec, missed = score(m)
    rows.append({"depth": depth, "mcw": mcw, "lr": lr, "iter": m.best_iteration,
                 "cost": c, "recall4": round(rec, 3), "missed4": missed})
    print(f"d={depth} mcw={mcw:>3} lr={lr} -> {c:>7,}  recall4={rec:.3f}  missed={missed}")

res = pd.DataFrame(rows).sort_values("cost")
print("\n", res.head(10).to_string(index=False))

d=3 mcw=  1 lr=0.05 ->  38,832  recall4=0.968  missed=13
d=3 mcw=  1 lr=0.1 ->  38,202  recall4=0.970  missed=12
d=3 mcw=  5 lr=0.05 ->  38,782  recall4=0.973  missed=11
d=3 mcw=  5 lr=0.1 ->  40,602  recall4=0.960  missed=16
d=3 mcw= 10 lr=0.05 ->  39,315  recall4=0.958  missed=17
d=3 mcw= 10 lr=0.1 ->  39,162  recall4=0.960  missed=16
d=3 mcw= 20 lr=0.05 ->  38,405  recall4=0.963  missed=15
d=3 mcw= 20 lr=0.1 ->  38,045  recall4=0.968  missed=13
d=3 mcw= 50 lr=0.05 ->  38,485  recall4=0.958  missed=17
d=3 mcw= 50 lr=0.1 ->  37,702  recall4=0.968  missed=13
d=4 mcw=  1 lr=0.05 ->  39,112  recall4=0.960  missed=16
d=4 mcw=  1 lr=0.1 ->  39,752  recall4=0.965  missed=14
d=4 mcw=  5 lr=0.05 ->  39,022  recall4=0.965  missed=14
d=4 mcw=  5 lr=0.1 ->  37,802  recall4=0.963  missed=15
d=4 mcw= 10 lr=0.05 ->  39,762  recall4=0.958  missed=17
d=4 mcw= 10 lr=0.1 ->  40,605  recall4=0.946  missed=22
d=4 mcw= 20 lr=0.05 ->  39,002  recall4=0.960  missed=16
d=4 mcw= 20 lr=0.1 ->  38,052  recall4=

In [28]:
best_d, best_m, best_lr = res.iloc[0][["depth", "mcw", "lr"]]
rows2 = []
for sub, col in product([0.6, 0.8, 1.0], [0.6, 0.8, 1.0]):
    m = xgb.XGBClassifier(
        objective="multi:softprob", num_class=5,
        max_depth=int(best_d), min_child_weight=int(best_m), learning_rate=float(best_lr),
        subsample=sub, colsample_bytree=col,
        n_estimators=2000, early_stopping_rounds=100,
        eval_metric="mlogloss", tree_method="hist", random_state=42, n_jobs=-1,
    )
    m.fit(Xlt, ylt, eval_set=[(Xl, yl)], verbose=False)
    c, rec, missed = score(m)
    rows2.append({"sub": sub, "col": col, "cost": c, "recall4": round(rec,3), "missed4": missed})
    print(f"sub={sub} col={col} -> {c:,}  recall4={rec:.3f}")

print(pd.DataFrame(rows2).sort_values("cost").to_string(index=False))

sub=0.6 col=0.6 -> 39,392  recall4=0.958
sub=0.6 col=0.8 -> 39,595  recall4=0.955
sub=0.6 col=1.0 -> 38,342  recall4=0.953
sub=0.8 col=0.6 -> 39,432  recall4=0.955
sub=0.8 col=0.8 -> 37,302  recall4=0.960
sub=0.8 col=1.0 -> 38,315  recall4=0.960
sub=1.0 col=0.6 -> 40,995  recall4=0.941
sub=1.0 col=0.8 -> 37,452  recall4=0.955
sub=1.0 col=1.0 -> 35,672  recall4=0.965
 sub  col  cost  recall4  missed4
 1.0  1.0 35672    0.965       14
 0.8  0.8 37302    0.960       16
 1.0  0.8 37452    0.955       18
 0.8  1.0 38315    0.960       16
 0.6  1.0 38342    0.953       19
 0.6  0.6 39392    0.958       17
 0.8  0.6 39432    0.955       18
 0.6  0.8 39595    0.955       18
 1.0  0.6 40995    0.941       24


In [29]:
final = xgb.XGBClassifier(
    objective="multi:softprob", num_class=5,
    max_depth=3, min_child_weight=50, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    n_estimators=2000, early_stopping_rounds=100,
    eval_metric="mlogloss", tree_method="hist",
    random_state=42, n_jobs=-1,
)
final.fit(Xlt, ylt, eval_set=[(Xl, yl)], verbose=False)

import joblib
joblib.dump(final, "../models/xgb_final.pkl")

pf = final.predict_proba(Xl)
pred_f = (pf @ COST).argmin(axis=1)
cm = confusion_matrix(yl, pred_f, labels=[0,1,2,3,4])
print("Validation cost:", f"{total_cost(yl, pred_f):,}")
print("Missed class-4:", cm[4,0], "| recall:", round(cm[4,4]/(yl==4).sum(), 3))
print("Calibration:", pf.mean(axis=0).round(4))

Validation cost: 37,702
Missed class-4: 13 | recall: 0.968
Calibration: [0.9051 0.0011 0.0032 0.0071 0.0836]


In [30]:
# --- Load test files ---
test_ops  = pd.read_csv(DATA_DIR + "test_operational_readouts.csv")
test_spec = pd.read_csv(DATA_DIR + "test_specifications.csv")
test_lab  = pd.read_csv(DATA_DIR + "test_labels.csv")

print("readouts:", test_ops.shape, "| specs:", test_spec.shape, "| labels:", test_lab.shape)
print(test_lab["class_label"].value_counts().sort_index())

readouts: (198140, 107) | specs: (5045, 9) | labels: (5045, 2)
class_label
0    4903
1      26
2      15
3      41
4      60
Name: count, dtype: int64


In [31]:
# --- Apply the identical feature pipeline ---
t = test_ops.sort_values(["vehicle_id", "time_step"]).reset_index(drop=True)
gt = t.groupby("vehicle_id")

t["dt"] = gt["time_step"].diff()
for c in counter_cols:
    d = gt[c].diff()
    t[f"{c}_reset"] = (d < 0).astype(int)
    t[f"{c}_rate"]  = d.clip(lower=0) / t["dt"]
t[rate_cols] = t[rate_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

for var in ["167", "272", "291", "158", "459", "397"]:
    cols = [c for c in test_ops.columns if c.startswith(f"{var}_")]
    tot = t[cols].sum(axis=1)
    t[[f"{c}_share" for c in cols]] = t[cols].div(tot.replace(0, np.nan), axis=0).fillna(0)

spec_enc = pd.get_dummies(test_spec, columns=[c for c in test_spec.columns if c != "vehicle_id"],
                          prefix_sep="=")
t = t.merge(spec_enc, on="vehicle_id", how="left")
t = t.copy()

# Align columns to training feature set (test may lack rare spec categories)
for col in feature_cols:
    if col not in t.columns:
        t[col] = 0
print("Missing spec categories filled:", sum(c not in test_ops.columns and c not in spec_enc.columns
                                              for c in spec_cols))

C:\Users\rajak\AppData\Local\Temp\ipykernel_43012\1682489372.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  t["dt"] = gt["time_step"].diff()
C:\Users\rajak\AppData\Local\Temp\ipykernel_43012\1682489372.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  t[f"{c}_reset"] = (d < 0).astype(int)
C:\Users\rajak\AppData\Local\Temp\ipykernel_43012\1682489372.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all co

Missing spec categories filled: 5


In [32]:
last_t = t.sort_values("time_step").groupby("vehicle_id").tail(1)
last_t = last_t.merge(test_lab, on="vehicle_id", how="inner")

X_test = last_t[feature_cols]
y_test = last_t["class_label"].values
print("Scoring:", X_test.shape)

p_test = final.predict_proba(X_test)
pred_test = (p_test @ COST).argmin(axis=1)

cm_t = confusion_matrix(y_test, pred_test, labels=[0,1,2,3,4])
tc = total_cost(y_test, pred_test)

print(f"\nTEST Total_cost = {tc:,}   (per vehicle: {tc/len(y_test):.2f})")
print("Always-0 baseline:", f"{total_cost(y_test, np.zeros_like(y_test)):,}")
print("\n", pd.DataFrame(cm_t, index=[f"actual {i}" for i in range(5)],
                          columns=[f"pred {i}" for i in range(5)]))
print("\nClass-4 recall:", round(cm_t[4,4]/max((y_test==4).sum(),1), 3), "| missed:", cm_t[4,0])
print("Calibration:", p_test.mean(axis=0).round(4))

Scoring: (5045, 212)

TEST Total_cost = 42,911   (per vehicle: 8.51)
Always-0 baseline: 56,100

           pred 0  pred 1  pred 2  pred 3  pred 4
actual 0     726       0       0       0    4177
actual 1       0       0       0       0      26
actual 2       0       0       0       0      15
actual 3       0       0       0       0      41
actual 4       1       0       0       0      59

Class-4 recall: 0.983 | missed: 1
Calibration: [0.851 0.002 0.006 0.013 0.128]


In [33]:
train_prior = pd.Series(ylt).value_counts(normalize=True).sort_index().values
true_prior  = np.array([0.972, 0.005, 0.003, 0.008, 0.012])   # from test label counts

p_adj = p_test * (true_prior / train_prior)
p_adj = p_adj / p_adj.sum(axis=1, keepdims=True)

pred_adj = (p_adj @ COST).argmin(axis=1)
cm_a = confusion_matrix(y_test, pred_adj, labels=[0,1,2,3,4])
print("Prior-corrected cost:", f"{total_cost(y_test, pred_adj):,}  (was 42,911)")
print("Flagged as 4:", (pred_adj == 4).sum(), "| missed class-4:", cm_a[4,0])
print("Calibration:", p_adj.mean(axis=0).round(4))

Prior-corrected cost: 36,056  (was 42,911)
Flagged as 4: 2619 | missed class-4: 7
Calibration: [0.9391 0.0099 0.007  0.018  0.026 ]


In [36]:
train_prior = pd.Series(ylt).value_counts(normalize=True).sort_index().values
val_prior   = pd.Series(yl).value_counts(normalize=True).sort_index().values

p_clean = p_test * (val_prior / train_prior)
p_clean = p_clean / p_clean.sum(axis=1, keepdims=True)

pred_clean = (p_clean @ COST).argmin(axis=1)
cm_c = confusion_matrix(y_test, pred_clean, labels=[0,1,2,3,4])
print("Clean prior-corrected cost:", f"{total_cost(y_test, pred_clean):,}")
print("Flagged as 4:", (pred_clean == 4).sum(), "| missed class-4:", cm_c[4,0])
print("Calibration:", p_clean.mean(axis=0).round(4))

Clean prior-corrected cost: 42,681
Flagged as 4: 4295 | missed class-4: 1
Calibration: [0.8527 0.0028 0.0045 0.0101 0.1299]


In [37]:
p_fail = p_clean[:, 1:].sum(axis=1)      # P(any warning class) per vehicle

rows = []
for budget in [0.02, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.52]:
    k = int(budget * len(p_fail))
    thresh = np.sort(p_fail)[-k]
    flagged = p_fail >= thresh
    caught  = ((y_test == 4) & flagged).sum()
    missed  = ((y_test == 4) & ~flagged).sum()
    rows.append({
        "alert_rate": f"{budget:.0%}",
        "trucks_flagged": k,
        "class4_caught": caught,
        "class4_missed": missed,
        "recall": round(caught / (y_test == 4).sum(), 3),
        "cost_if_10_500": k * 10 + missed * 500,
    })

print(pd.DataFrame(rows).to_string(index=False))

alert_rate  trucks_flagged  class4_caught  class4_missed  recall  cost_if_10_500
        2%             100              5             55   0.083           28500
        5%             252             11             49   0.183           27020
       10%             504             18             42   0.300           26040
       15%             756             25             35   0.417           25060
       20%            1009             30             30   0.500           25090
       30%            1513             37             23   0.617           26630
       40%            2018             45             15   0.750           27680
       52%            2623             52              8   0.867           30230


In [38]:
p_fail = p_clean[:, 1:].sum(axis=1)          # risk score per truck

rows = []
for budget in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60, 0.85]:
    k = int(budget * len(p_fail))
    thresh = np.sort(p_fail)[-k]
    pred = np.where(p_fail >= thresh, 4, 0)
    cm = confusion_matrix(y_test, pred, labels=[0,1,2,3,4])
    rows.append({
        "alert_rate": f"{budget:.0%}",
        "inspected": k,
        "TOTAL_COST": total_cost(y_test, pred),
        "class4_caught": cm[4,4],
        "class4_missed": cm[4,0],
    })

print(pd.DataFrame(rows).to_string(index=False))
print("\nCurrent model:", f"{total_cost(y_test, pred_clean):,}  (inspects 4,295)")
print("Do nothing:     56,100")

alert_rate  inspected  TOTAL_COST  class4_caught  class4_missed
        5%        252       49588             11             49
       10%        504       46022             18             42
       15%        756       42353             25             35
       20%       1009       40118             30             30
       25%       1261       37886             33             27
       30%       1513       37057             37             23
       40%       2018       34906             45             15
       50%       2522       35476             51              9
       60%       3027       36277             54              6
       85%       4288       42611             59              1

Current model: 42,681  (inspects 4,295)
Do nothing:     56,100


In [39]:
# --- Choose operating point on VALIDATION ---
val_prior_arr = pd.Series(yl).value_counts(normalize=True).sort_index().values
pv = final.predict_proba(Xl)
pv = pv * (val_prior_arr / train_prior)
pv = pv / pv.sum(axis=1, keepdims=True)

def curve(score, y, label):
    n4 = (y == 4).sum()
    out = []
    for b in np.arange(0.05, 0.95, 0.05):
        k = int(b * len(score))
        thresh = np.sort(score)[-k]
        pred = np.where(score >= thresh, 4, 0)
        cm = confusion_matrix(y, pred, labels=[0,1,2,3,4])
        out.append({"score": label, "alert_rate": round(b, 2), "k": k,
                    "recall4": round(cm[4,4]/n4, 3), "missed4": cm[4,0],
                    "cost": total_cost(y, pred), "thresh": thresh})
    return pd.DataFrame(out)

c_sum = curve(pv[:, 1:].sum(axis=1), yl, "P(1..4)")
c_c4  = curve(pv[:, 4],              yl, "P(4)")
both = pd.concat([c_sum, c_c4])

# Which scoring function catches more failures at the same budget?
print(both.pivot(index="alert_rate", columns="score", values="recall4").to_string())

score       P(1..4)   P(4)
alert_rate                
0.05          0.287  0.282
0.10          0.421  0.426
0.15          0.510  0.512
0.20          0.599  0.601
0.25          0.673  0.668
0.30          0.733  0.730
0.35          0.787  0.780
0.40          0.822  0.824
0.45          0.854  0.851
0.50          0.889  0.884
0.55          0.911  0.913
0.60          0.921  0.921
0.65          0.943  0.941
0.70          0.955  0.958
0.75          0.968  0.968
0.80          0.973  0.973
0.85          0.978  0.975
0.90          0.993  0.990


In [40]:
TARGET_RECALL = 0.90

feasible = both[both["recall4"] >= TARGET_RECALL]
best = feasible.sort_values("cost").iloc[0]
print(f"\nBest config meeting {TARGET_RECALL:.0%} recall on validation:")
print(best.to_string())

# --- Apply the chosen threshold UNCHANGED to test ---
score_test = p_clean[:, 4] if best["score"] == "P(4)" else p_clean[:, 1:].sum(axis=1)
pred_test_final = np.where(score_test >= best["thresh"], 4, 0)

cm_f = confusion_matrix(y_test, pred_test_final, labels=[0,1,2,3,4])
print("\n=== TEST (threshold chosen on validation, applied unchanged) ===")
print("Cost:", f"{total_cost(y_test, pred_test_final):,}   vs baseline 56,100")
print("Inspected:", int(pred_test_final.sum()/4), "of", len(y_test))
print("Class-4 recall:", round(cm_f[4,4]/(y_test==4).sum(), 3), "| missed:", cm_f[4,0])


Best config meeting 90% recall on validation:
score          P(1..4)
alert_rate        0.75
k                 3532
recall4          0.968
missed4             13
cost             37802
thresh        0.020373

=== TEST (threshold chosen on validation, applied unchanged) ===
Cost: 43,041   vs baseline 56,100
Inspected: 4331 of 5045
Class-4 recall: 0.983 | missed: 1


In [41]:
print(feasible.sort_values("alert_rate")[["score","alert_rate","k","recall4","missed4","cost"]].to_string(index=False))

  score  alert_rate    k  recall4  missed4  cost
P(1..4)        0.55 2590    0.911       36 40918
   P(4)        0.55 2590    0.913       35 40408
   P(4)        0.60 2826    0.921       32 41238
P(1..4)        0.60 2826    0.921       32 41238
P(1..4)        0.65 3061    0.943       23 38998
   P(4)        0.65 3061    0.941       24 39508
   P(4)        0.70 3297    0.958       17 37895
P(1..4)        0.70 3297    0.955       18 38002
   P(4)        0.75 3532    0.968       13 37802
P(1..4)        0.75 3532    0.968       13 37802
P(1..4)        0.80 3768    0.973       11 39142
   P(4)        0.80 3768    0.973       11 39142
   P(4)        0.85 4003    0.975       10 40982
P(1..4)        0.85 4003    0.978        9 40472
P(1..4)        0.90 4239    0.993        3 39772
   P(4)        0.90 4239    0.990        4 40282


In [42]:
TARGET_RECALL = 0.90
feasible = both[(both["score"] == "P(4)") & (both["recall4"] >= TARGET_RECALL)]
best = feasible.sort_values("alert_rate").iloc[0]      # smallest budget that meets the constraint
print(best.to_string())

score_test = p_clean[:, 4]
pred_final = np.where(score_test >= best["thresh"], 4, 0)
cm_f = confusion_matrix(y_test, pred_final, labels=[0,1,2,3,4])
print("\nTEST cost:", f"{total_cost(y_test, pred_final):,}  vs 56,100")
print("Inspected:", int((pred_final == 4).sum()), "of", len(y_test))
print("Recall:", round(cm_f[4,4]/(y_test==4).sum(), 3), "| missed:", cm_f[4,0])

score             P(4)
alert_rate        0.55
k                 2590
recall4          0.913
missed4             35
cost             40408
thresh        0.034989

TEST cost: 37,684  vs 56,100
Inspected: 3563 of 5045
Recall: 0.967 | missed: 2


In [43]:
best = both[(both["score"] == "P(4)") & (both["alert_rate"] == 0.55)].iloc[0]
print("Chosen threshold:", round(float(best["thresh"]), 6))

score_test = p_clean[:, 4]
pred_final = np.where(score_test >= best["thresh"], 4, 0)
cm_f = confusion_matrix(y_test, pred_final, labels=[0,1,2,3,4])

print("\n=== FINAL (validation-selected, applied unchanged to test) ===")
print("Cost:", f"{total_cost(y_test, pred_final):,}  vs baseline 56,100")
print("Reduction:", f"{100*(1 - total_cost(y_test, pred_final)/56100):.1f}%")
print("Inspected:", int((pred_final == 4).sum()), "of", len(y_test))
print("Class-4 recall:", round(cm_f[4,4]/(y_test==4).sum(), 3), "| missed:", cm_f[4,0])

Chosen threshold: 0.034989

=== FINAL (validation-selected, applied unchanged to test) ===
Cost: 37,684  vs baseline 56,100
Reduction: 32.8%
Inspected: 3563 of 5045
Class-4 recall: 0.967 | missed: 2


In [44]:
flagged = pred_final == 4

print("Class | n_vehicles | flagged | flag_rate")
for c in range(5):
    mask = (y_test == c)
    print(f"  {c}   |    {mask.sum():>5}    |  {(mask & flagged).sum():>5}  |  {(mask & flagged).sum()/max(mask.sum(),1):.3f}")

print("\nFull confusion matrix:")
print(pd.DataFrame(confusion_matrix(y_test, pred_final, labels=[0,1,2,3,4]),
                   index=[f"actual {i}" for i in range(5)],
                   columns=[f"pred {i}" for i in range(5)]))

Class | n_vehicles | flagged | flag_rate
  0   |     4903    |   3428  |  0.699
  1   |       26    |     26  |  1.000
  2   |       15    |     13  |  0.867
  3   |       41    |     38  |  0.927
  4   |       60    |     58  |  0.967

Full confusion matrix:
          pred 0  pred 1  pred 2  pred 3  pred 4
actual 0    1475       0       0       0    3428
actual 1       0       0       0       0      26
actual 2       2       0       0       0      13
actual 3       3       0       0       0      38
actual 4       2       0       0       0      58


In [ ]:
pred_5class = np.where(flagged, (p_clean @ COST).argmin(axis=1), 0)
pred_5class = np.where(flagged & (pred_5class == 0), 4, pred_5class)   # flagged trucks get at least a warning
print(classification_report(y_test, pred_5class, zero_division=0))
print("Cost:", f"{total_cost(y_test, pred_5class):,}")

              precision    recall  f1-score   support

           0       1.00      0.30      0.46      4903
           1       0.00      0.00      0.00        26
           2       0.00      0.00      0.00        15
           3       0.00      0.00      0.00        41
           4       0.02      0.97      0.03        60

    accuracy                           0.30      5045
   macro avg       0.20      0.25      0.10      5045
weighted avg       0.97      0.30      0.45      5045

Cost: 37,684


: 